# Notebook 04 — Model Training
### NER System | Sports & Political News

Train two models:
- **Model 1**: SpaCy NER (neural deep learning)
- **Model 2**: CRF — Conditional Random Field (classical ML)

In [1]:
import pandas as pd
import numpy as np
import json, os, random, warnings
warnings.filterwarnings('ignore')
print('Libraries loaded OK')

Libraries loaded OK


## Step 1 — Load Data

In [2]:
df = pd.read_csv('../data/cleaned_data.csv')
with open('../data/iob_data.json') as f:
    iob_data = json.load(f)
with open('../data/spacy_training_data.json') as f:
    spacy_raw = json.load(f)

print(f'Cleaned data rows  : {len(df)}')
print(f'IOB records        : {len(iob_data)}')
print(f'SpaCy records      : {len(spacy_raw)}')

Cleaned data rows  : 1737
IOB records        : 1737
SpaCy records      : 1737


## Step 2 — Train / Test Split (80/20)

In [3]:
from sklearn.model_selection import train_test_split

# Split IOB data
iob_train, iob_test = train_test_split(iob_data, test_size=0.2, random_state=42)

# Split SpaCy data
spacy_data = [(d['text'], {'entities':[(e[0],e[1],e[2]) for e in d['entities']]}) for d in spacy_raw]
spacy_train, spacy_test = train_test_split(spacy_data, test_size=0.2, random_state=42)

print(f'Train set : {len(iob_train)} records')
print(f'Test set  : {len(iob_test)} records')

Train set : 1389 records
Test set  : 348 records


## Step 3 — Model 1: Train SpaCy NER

In [4]:
import spacy
from spacy.training import Example
from spacy.util import minibatch, compounding

print('Training SpaCy NER model...')
nlp_spacy = spacy.load('en_core_web_sm')
ner = nlp_spacy.get_pipe('ner')
ner.add_label('PERSON')
ner.add_label('LOCATION')

optimizer = nlp_spacy.initialize()
losses_log = []

for itn in range(30):
    random.shuffle(spacy_train)
    losses = {}
    batches = minibatch(spacy_train, size=compounding(4.0, 32.0, 1.001))
    for batch in batches:
        examples = []
        for text, annots in batch:
            try:
                doc = nlp_spacy.make_doc(text)
                eg  = Example.from_dict(doc, annots)
                examples.append(eg)
            except:
                pass
        if examples:
            nlp_spacy.update(examples, sgd=optimizer, losses=losses)
    loss_val = round(losses.get('ner',0), 4)
    losses_log.append(loss_val)
    if (itn+1) % 5 == 0:
        print(f'  Iteration {itn+1:3d} | Loss: {loss_val}')

os.makedirs('../models', exist_ok=True)
nlp_spacy.to_disk('../models/spacy_model')
print('SpaCy model saved to ../models/spacy_model')

Training SpaCy NER model...
  Iteration   5 | Loss: 116.8088
  Iteration  10 | Loss: 53.7178
  Iteration  15 | Loss: 32.4593
  Iteration  20 | Loss: 40.5267
  Iteration  25 | Loss: 27.0896
  Iteration  30 | Loss: 35.2512
SpaCy model saved to ../models/spacy_model


## Step 4 — Model 2: Train CRF

In [5]:
import sklearn_crfsuite

def word_features(tokens, i):
    word = tokens[i]
    f = {
        'word.lower'      : word.lower(),
        'word.isupper'    : word.isupper(),
        'word.istitle'    : word.istitle(),
        'word.isdigit'    : word.isdigit(),
        'word.prefix2'    : word[:2].lower(),
        'word.prefix3'    : word[:3].lower(),
        'word.suffix2'    : word[-2:].lower(),
        'word.suffix3'    : word[-3:].lower(),
        'word.has_hyphen' : '-' in word,
        'word.has_digit'  : any(c.isdigit() for c in word),
        'word.length'     : len(word),
        'BOS'             : i == 0,
        'EOS'             : i == len(tokens)-1,
    }
    if i > 0:
        p = tokens[i-1]
        f.update({'prev.lower':p.lower(),'prev.istitle':p.istitle(),'prev.isupper':p.isupper()})
    if i < len(tokens)-1:
        n = tokens[i+1]
        f.update({'next.lower':n.lower(),'next.istitle':n.istitle(),'next.isupper':n.isupper()})
    return f

def sent_features(tokens):
    return [word_features(tokens, i) for i in range(len(tokens))]

X_train = [sent_features(r['tokens']) for r in iob_train]
y_train = [r['tags'] for r in iob_train]
X_test  = [sent_features(r['tokens']) for r in iob_test]
y_test  = [r['tags'] for r in iob_test]

print('Training CRF model...')
crf = sklearn_crfsuite.CRF(algorithm='lbfgs', c1=0.1, c2=0.1, max_iterations=200, all_possible_transitions=True)
crf.fit(X_train, y_train)

import joblib
joblib.dump(crf, '../models/crf_model.pkl')
print('CRF model saved to ../models/crf_model.pkl')

Training CRF model...
CRF model saved to ../models/crf_model.pkl


## Step 5 — Evaluate Both Models

In [6]:
from sklearn.metrics import classification_report
from sklearn_crfsuite.metrics import flat_classification_report

labels = ['B-PERSON','I-PERSON','B-LOCATION','I-LOCATION']

# ── SpaCy Evaluation ────────────────────────────────────
print('='*55)
print('MODEL 1: SpaCy NER — Evaluation')
print('='*55)
nlp_loaded = spacy.load('../models/spacy_model')

y_true_spacy, y_pred_spacy = [], []
for text, annots in spacy_test:
    doc = nlp_loaded(text)
    true_ents = {(e[0],e[1]):e[2] for e in annots['entities']}
    pred_ents = {(e.start_char,e.end_char):e.label_ for e in doc.ents}
    for span in set(list(true_ents.keys())+list(pred_ents.keys())):
        y_true_spacy.append(true_ents.get(span,'O'))
        y_pred_spacy.append(pred_ents.get(span,'O'))

spacy_labels = ['PERSON','LOCATION']
print(classification_report(y_true_spacy, y_pred_spacy, labels=spacy_labels, zero_division=0))

# ── CRF Evaluation ──────────────────────────────────────
print('='*55)
print('MODEL 2: CRF — Evaluation')
print('='*55)
y_pred_crf = crf.predict(X_test)
print(flat_classification_report(y_test, y_pred_crf, labels=labels, zero_division=0))

MODEL 1: SpaCy NER — Evaluation


d:\NER\venv311\Lib\site-packages\spacy\pipeline\attributeruler.py:149: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  matches = self.matcher(doc, allow_missing=True, as_spans=False)


              precision    recall  f1-score   support

      PERSON       0.59      0.26      0.37       159
    LOCATION       0.78      0.60      0.68       152

   micro avg       0.71      0.43      0.53       311
   macro avg       0.68      0.43      0.52       311
weighted avg       0.68      0.43      0.52       311

MODEL 2: CRF — Evaluation
              precision    recall  f1-score   support

    B-PERSON       0.82      0.65      0.73       159
    I-PERSON       0.74      0.67      0.70        99
  B-LOCATION       0.95      0.75      0.84       154
  I-LOCATION       0.94      0.68      0.79        22

   micro avg       0.85      0.69      0.76       434
   macro avg       0.86      0.69      0.76       434
weighted avg       0.85      0.69      0.76       434



## Step 6 — Save Metrics

In [7]:
from sklearn.metrics import precision_recall_fscore_support

# SpaCy metrics
metrics = {}
for label in ['PERSON','LOCATION']:
    binary_true = [1 if l==label else 0 for l in y_true_spacy]
    binary_pred = [1 if l==label else 0 for l in y_pred_spacy]
    p,r,f,_ = precision_recall_fscore_support(binary_true, binary_pred, average='binary', zero_division=0)
    metrics[f'spacy_{label.lower()}'] = {'precision':round(p,3),'recall':round(r,3),'f1':round(f,3)}

# CRF metrics
y_test_flat  = [t for s in y_test for t in s]
y_pred_flat  = [t for s in y_pred_crf for t in s]
for label in ['B-PERSON','B-LOCATION']:
    short = label.replace('B-','').lower()
    binary_true = [1 if l==label else 0 for l in y_test_flat]
    binary_pred = [1 if l==label else 0 for l in y_pred_flat]
    p,r,f,_ = precision_recall_fscore_support(binary_true, binary_pred, average='binary', zero_division=0)
    metrics[f'crf_{short}'] = {'precision':round(p,3),'recall':round(r,3),'f1':round(f,3)}

metrics['spacy_losses'] = losses_log
with open('../models/metrics.json','w') as f:
    json.dump(metrics, f, indent=2)

print('Metrics saved to ../models/metrics.json')
print('\n=== FINAL COMPARISON ===')
print(f'{"Model":8s} {"Entity":12s} {"Precision":12s} {"Recall":10s} {"F1":8s}')
print('-'*52)
for key, vals in metrics.items():
    if key == 'spacy_losses': continue
    model, ent = key.split('_')
    print(f'{model.upper():8s} {ent.upper():12s} {vals["precision"]:12.3f} {vals["recall"]:10.3f} {vals["f1"]:8.3f}')

Metrics saved to ../models/metrics.json

=== FINAL COMPARISON ===
Model    Entity       Precision    Recall     F1      
----------------------------------------------------
SPACY    PERSON              0.592      0.264    0.365
SPACY    LOCATION            0.778      0.599    0.677
CRF      PERSON              0.819      0.654    0.727
CRF      LOCATION            0.950      0.747    0.836


In [8]:
from sklearn.metrics import accuracy_score
from sklearn_crfsuite.metrics import flat_accuracy_score

# ── SpaCy Token-level Accuracy ────────────────────────────────────────────
correct = 0
total   = 0
for text, annots in spacy_test:
    doc = nlp_loaded(text)
    pred_ents = {(e.start_char, e.end_char): e.label_ for e in doc.ents}
    true_ents = {(e[0], e[1]): e[2] for e in annots['entities']}
    all_spans = set(list(pred_ents.keys()) + list(true_ents.keys()))
    for span in all_spans:
        total += 1
        if pred_ents.get(span) == true_ents.get(span):
            correct += 1

spacy_accuracy = round(correct / total * 100, 2) if total > 0 else 0

# ── CRF Token-level Accuracy ──────────────────────────────────────────────
crf_accuracy = round(flat_accuracy_score(y_test, y_pred_crf) * 100, 2)

print(f'SpaCy NER Accuracy : {spacy_accuracy}%')
print(f'CRF Accuracy       : {crf_accuracy}%')

# Save to metrics
metrics['spacy_accuracy'] = spacy_accuracy
metrics['crf_accuracy']   = crf_accuracy
with open('../models/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Accuracy saved to metrics.json')

d:\NER\venv311\Lib\site-packages\spacy\pipeline\attributeruler.py:149: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  matches = self.matcher(doc, allow_missing=True, as_spans=False)


SpaCy NER Accuracy : 36.74%
CRF Accuracy       : 97.73%
Accuracy saved to metrics.json


---
## Notebook 04 Complete ✅
**Next → Notebook 05: Evaluation**